# Part 3 — Entity resolution

Read the approved `gold_*` observations, establish Australian business identities, then conservatively link observations without identifiers. Source claims remain in the gold tables. Profiles and reconciliation belong in Part 4.

`entities_pool` contains only `entity_id`, `abn`, and `acn`. Stated ABNs become entity IDs. An ACN-only observation can use its checksum-derived identity value; its `abn` remains null until a source states it. This distinguishes a derived identity from evidence of ABN registration.

Identifier algorithms: [ABN Lookup](https://abr.business.gov.au/Help/AbnFormat), [ASIC ACN check digit](https://asic.gov.au/for-business/registering-a-company/steps-to-register-a-company/australian-company-numbers/australian-company-number-digit-check/), [ASIC ACN/ABN relationship](https://asic.gov.au/for-business/small-business/protecting-your-small-business/dealing-with-businesses-and-companies/).

Run in order. The matching scores are decision rules, not calibrated probabilities. Ambiguous, conflicting, or weak evidence goes to review.

The pasted ABN example `51824753556` is valid, but its suffix `824753556` fails the ACN checksum. The derivation example used here is validated ACN `004085616` → identity `53004085616`.

## Imports and settings

In [1]:
import hashlib
import json
import re
import unicodedata
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urlparse

import duckdb
from firmable.database import connect_database, close_database
import pandas as pd
from IPython.display import display
from rapidfuzz.fuzz import ratio
from rapidfuzz import process

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DB_PATH = PROJECT_ROOT / "firmable.duckdb"
GOLD_COPY_DIR = PROJECT_ROOT / "outputs" / "part2" / "gold_inspection"
PART3_DIR = PROJECT_ROOT / "outputs" / "part3"
PART3_DIR.mkdir(parents=True, exist_ok=True)

MIN_LINK_SCORE = 0.97
MIN_SCORE_MARGIN = 0.02
MAX_MATCH_PASSES = 10
MAX_CANDIDATES = 500
REVIEW_SAMPLE_SIZE = 50
REVIEW_SEED = 20261001
REVIEW_PATH = PART3_DIR / "link_review_50.csv"

SHOW_DETAILS = False
PREVIEW_ROWS = 5


def stage(name):
    print(f"\nPART 3 — {name}")


def dataset_label(name):
    return str(name).removeprefix("gold_").replace("_", " ")


DATABASE_READY = False


## Load gold observations

In [2]:
def quote_identifier(value):
    return '"' + str(value).replace('"', '""') + '"'


def table_reference(schema, table):
    return f"{quote_identifier(schema)}.{quote_identifier(table)}"


def load_part3_sources():
    global DATABASE_MODE
    connection = connect_database(DB_PATH) if DB_PATH.exists() else None
    DATABASE_MODE = "live"
    if connection is None:
        copies = sorted(GOLD_COPY_DIR.glob("gold_*.parquet"))
        if not copies:
            print("Part 3 paused: database unavailable and no exported gold files. Existing results are preserved.")
            return None
        connection = duckdb.connect(":memory:")
        DATABASE_MODE = "gold_copies"
        try:
            for path in copies:
                quoted_path = str(path).replace("'", "''")
                connection.execute(
                    f"CREATE VIEW {quote_identifier(path.stem)} AS SELECT * FROM read_parquet('{quoted_path}')"
                )
        except duckdb.Error as error:
            print("Part 3 paused: exported gold files could not be read:", error)
            close_database(connection)
            return None
        print("Reading exported gold files; database persistence will be retried when saving.")
    try:
        tables = connection.execute("""
            SELECT table_schema, table_name FROM information_schema.tables
            WHERE starts_with(table_name, 'gold_') ORDER BY table_schema, table_name
        """).fetchall()
        if not tables:
            print("Part 3 paused: no approved gold tables. Existing results are preserved.")
            return None
        frames = []
        for schema, table in tables:
            frame = connection.execute(f"SELECT * FROM {table_reference(schema, table)}").df()
            frame["gold_table"] = table
            frames.append(frame)
        return pd.concat(frames, ignore_index=True)
    except duckdb.Error as error:
        print("Part 3 source read stopped safely:", error)
        return None
    finally:
        close_database(connection)


close_database(globals().get("part3_con"))
part3_con = None
stage("Load source observations")
gold_observations = load_part3_sources()
DATABASE_READY = gold_observations is not None
if DATABASE_READY:
    print("Reading:", "Firmable database" if DATABASE_MODE == "live" else "exported gold copies")
    for table, rows in gold_observations.groupby("gold_table").size().items():
        print(f"  {dataset_label(table)}: {rows:,} records")
    print(f"Total: {len(gold_observations):,} records")



PART 3 — Load source observations
Reading: Firmable database
  abn bulk extract: 1,000 records
  acnc registered charities: 1,000 records
  asic company: 1,000 records
  business establishments location and industry classification: 1,000 records
  victorian government schools abns: 1,000 records
  victorian liquor licences by location: 1,000 records
Total: 6,000 records


## Identifier validation and normalization

In [3]:
ABN_WEIGHTS = (10, 1, 3, 5, 7, 9, 11, 13, 15, 17, 19)
ACN_WEIGHTS = (8, 7, 6, 5, 4, 3, 2, 1)


def text_value(value):
    if value is None or pd.isna(value):
        return None
    value = str(value).strip()
    return None if value.casefold() in {"", "none", "null", "nan", "n/a", "unknown"} else value


def normalize_identifier(value):
    value = text_value(value)
    if value is None:
        return None
    # Formatting only. Never pad digits or remove letters to manufacture an ID.
    return re.sub(r"[\s-]", "", value)


def valid_abn(value):
    if not value or not re.fullmatch(r"[0-9]{11}", value) or set(value) == {"0"}:
        return False
    digits = [int(digit) for digit in value]
    digits[0] -= 1
    return sum(digit * weight for digit, weight in zip(digits, ABN_WEIGHTS)) % 89 == 0


def valid_acn(value):
    if not value or not re.fullmatch(r"[0-9]{9}", value) or set(value) == {"0"}:
        return False
    total = sum(int(digit) * weight for digit, weight in zip(value[:8], ACN_WEIGHTS))
    return (10 - total % 10) % 10 == int(value[-1])


def abn_candidates_from_acn(acn):
    if not valid_acn(acn):
        return []
    return [f"{prefix:02d}{acn}" for prefix in range(10, 100) if valid_abn(f"{prefix:02d}{acn}")]


def derive_abn(acn):
    candidates = abn_candidates_from_acn(acn)
    # The checksum can admit two prefixes. Without stated evidence, abstain.
    return candidates[0] if len(candidates) == 1 else None


assert valid_abn("51824753556")
assert valid_acn("004085616")
assert not valid_acn("824753556")  # The ABN example suffix is not a valid ACN.
assert derive_abn("004085616") == "53004085616"


## Matching features

In [4]:
FEATURE_COLUMNS = {
    "legal_name": "entity_legal_name",
    "trading_name": "entity_trading_name",
    "website": "entity_website",
    "address": "address_full",
    "locality": "address_locality",
    "state": "address_state",
    "postcode": "address_postcode",
    "country": "address_country",
}
SHARED_WEBSITE_HOSTS = {
    "facebook.com", "instagram.com", "linkedin.com", "youtube.com", "linktr.ee",
    "wordpress.com", "wixsite.com", "weebly.com", "sites.google.com",
}


def normalize_text(value):
    value = text_value(value)
    if value is None:
        return None
    value = unicodedata.normalize("NFKC", value).upper().replace("&", " AND ")
    value = re.sub(r"[^\w\s]", " ", value).replace("_", " ")
    value = re.sub(r"\s+", " ", value).strip()
    return value or None


def normalize_name(value):
    value = normalize_text(value)
    if value:
        value = re.sub(r"\bPTY\b", "PROPRIETARY", value)
        value = re.sub(r"\bLTD\b", "LIMITED", value)
    return value


def website_host(value):
    value = text_value(value)
    if not value:
        return None
    parsed = urlparse(value if "://" in value else "https://" + value)
    host = (parsed.hostname or "").lower().rstrip(".")
    if host.startswith("www."):
        host = host[4:]
    if "." not in host or any(host == shared or host.endswith("." + shared) for shared in SHARED_WEBSITE_HOSTS):
        return None
    return host


def matching_features(row):
    features = {}
    for field, column in FEATURE_COLUMNS.items():
        value = row.get(column)
        if field in {"legal_name", "trading_name"}:
            features[field] = normalize_name(value)
        elif field == "website":
            features[field] = website_host(value)
        else:
            features[field] = normalize_text(value)
    return features


def feature_confidence(row, field):
    key = FEATURE_COLUMNS[field].replace("entity_", "entity.", 1).replace("address_", "address.", 1)
    try:
        values = json.loads(text_value(row.get("field_confidence")) or "{}")
        return float(values.get(key, 1.0))
    except (ValueError, TypeError):
        return 0.0


def prepare_observations(frame):
    observations = []
    for row_number, row in enumerate(frame.to_dict("records")):
        source_id = text_value(row.get("observation_source_id"))
        record_id = text_value(row.get("observation_source_record_id"))
        identity = [source_id, record_id] if source_id and record_id else [source_id, record_id, row.get("gold_table"), row_number]
        key = hashlib.sha256(json.dumps(identity).encode()).hexdigest()
        abn = normalize_identifier(row.get("entity_abn"))
        acn = normalize_identifier(row.get("entity_acn"))
        problems = []
        if not source_id or not record_id:
            problems.append("missing_source_key")
        if abn and not valid_abn(abn):
            problems.append("invalid_stated_abn")
        if acn and not valid_acn(acn):
            problems.append("invalid_stated_acn")
        if valid_abn(abn) and valid_acn(acn) and abn[-9:] != acn:
            problems.append("abn_acn_conflict")
        features = matching_features(row)
        # Unsupported/low-confidence mapped fields cannot contribute matching evidence.
        for field in features:
            if feature_confidence(row, field) < 0.85:
                features[field] = None
        claim = {key: text_value(value) for key, value in row.items()
                 if key.startswith(("entity_", "address_"))}
        observations.append({
            "observation_id": key, "source_id": source_id, "source_record_id": record_id,
            "gold_table": row.get("gold_table"), "row": row,
            "abn": abn, "acn": acn, "features": features, "problems": problems,
            "claim": claim,
        })

    grouped = defaultdict(list)
    for observation in observations:
        grouped[observation["observation_id"]].append(observation)
    unique = []
    duplicates = 0
    for group in grouped.values():
        first = group[0]
        duplicates += len(group) - 1
        if len({json.dumps(item["claim"], sort_keys=True) for item in group}) > 1:
            first["problems"].append("conflicting_duplicate_source_record")
        first["gold_table"] = ", ".join(sorted({str(item["gold_table"]) for item in group}))
        unique.append(first)
    return sorted(unique, key=lambda item: item["observation_id"]), duplicates


## Identifier links and entity registry

In [5]:
LINK_COLUMNS = [
    "link_id", "source_id", "source_record_id", "entity_id", "match_method",
    "match_evidence", "link_confidence", "created_at",
]


def make_link(observation, entity_id, method, score, evidence, pass_number):
    evidence = {
        **evidence,
        "pass": pass_number,
        "stated_abn": observation["abn"],
        "stated_acn": observation["acn"],
        "abn_origin": "stated" if valid_abn(observation["abn"]) else "derived" if valid_acn(observation["acn"]) else "entity_match",
        "acn_origin": "stated" if valid_acn(observation["acn"]) else "not_stated",
    }
    return {
        "link_id": observation["observation_id"],
        "source_id": observation["source_id"],
        "source_record_id": observation["source_record_id"],
        "entity_id": entity_id, "match_method": method,
        "match_evidence": json.dumps(evidence, sort_keys=True),
        "link_confidence": float(score),
        "created_at": datetime.now(timezone.utc).isoformat(),
    }


def seed_identifiers(observations):
    pool = {}
    links = {}
    issues = {item["observation_id"]: "; ".join(item["problems"])
              for item in observations if item["problems"]}
    acn_claims = defaultdict(set)
    for item in observations:
        if not item["problems"] and valid_abn(item["abn"]) and valid_acn(item["acn"]):
            acn_claims[item["acn"]].add(item["abn"])
    conflicting_acns = {acn for acn, abns in acn_claims.items() if len(abns) > 1}

    # Seed stated ABNs first; order cannot affect an ACN-only observation's identity.
    for item in observations:
        key = item["observation_id"]
        if key in issues:
            continue
        if item["acn"] in conflicting_acns:
            issues[key] = "acn_claimed_by_multiple_abns"
            continue
        if not valid_abn(item["abn"]):
            continue
        entity_id = item["abn"]
        entity = pool.setdefault(entity_id, {"entity_id": entity_id, "abn": entity_id, "acn": None})
        if valid_acn(item["acn"]):
            entity["acn"] = item["acn"]
        links[key] = make_link(item, entity_id, "abn_exact", 1.0, {"conflicts": []}, 1)

    acn_index = {entity["acn"]: entity_id for entity_id, entity in pool.items() if entity["acn"]}
    stated_types = defaultdict(set)
    for item in observations:
        if item["observation_id"] in links:
            kind = text_value(item["row"].get("entity_entity_type"))
            if kind and kind != "unknown":
                stated_types[links[item["observation_id"]]["entity_id"]].add(kind)

    for item in observations:
        key = item["observation_id"]
        if key in issues or key in links or not valid_acn(item["acn"]):
            continue
        if item["acn"] in conflicting_acns:
            issues[key] = "acn_claimed_by_multiple_abns"
            continue
        known_entity = acn_index.get(item["acn"])
        entity_id = known_entity or derive_abn(item["acn"])
        if entity_id is None:
            issues[key] = "ambiguous_acn_abn_prefix"
            continue
        incompatible = stated_types[entity_id] - {"company"}
        if incompatible:
            issues[key] = "acn_derived_identity_conflicts_with_entity_type"
            continue
        entity = pool.setdefault(entity_id, {"entity_id": entity_id, "abn": None, "acn": item["acn"]})
        if entity["acn"] not in {None, item["acn"]}:
            issues[key] = "entity_has_different_acn"
            continue
        entity["acn"] = item["acn"]
        acn_index[item["acn"]] = entity_id
        method = "acn_exact" if known_entity else "acn_derived_abn"
        links[key] = make_link(item, entity_id, method, 1.0, {
            "derived_identity": derive_abn(item["acn"]),
            "abn_registration_confirmed_by_source": entity["abn"] is not None,
            "conflicts": [],
        }, 1)
    return pool, links, issues


## Match records without ABN or ACN

Already-linked source observations provide names, websites, and addresses for each entity. The matcher compares an unlinked record with that evidence.

Automatic linking requires a score of at least `0.97`, enough separation from competing entities, and no detected conflict. Name alone cannot create an automatic link. All scores are decision rules, not measured probabilities.

A separate possible-match report also searches similar names across existing entities. Its weaker suggestions are for human review only; they do not change `entity_link_index`.

In [6]:
def build_matching_index(observations, links):
    profiles = defaultdict(lambda: defaultdict(dict))
    blocks = defaultdict(set)
    for item in observations:
        link = links.get(item["observation_id"])
        if link is None:
            continue
        entity_id = link["entity_id"]
        for field, value in item["features"].items():
            if not value:
                continue
            support = {"source_id": item["source_id"], "source_record_id": item["source_record_id"],
                       "link_confidence": link["link_confidence"]}
            previous = profiles[entity_id][field].get(value)
            if previous is None or previous["link_confidence"] < support["link_confidence"]:
                profiles[entity_id][field][value] = support
            blocks[(field, value)].add(entity_id)
            if field in {"legal_name", "trading_name"}:
                blocks[("name", value)].add(entity_id)
    return profiles, blocks


def candidate_entities(item, blocks):
    candidates = set()
    for field, value in item["features"].items():
        if not value:
            continue
        if field in {"legal_name", "trading_name"}:
            candidates.update(blocks.get(("name", value), set()))
        elif field in {"website", "address", "postcode"}:
            candidates.update(blocks.get((field, value), set()))
    return candidates


def score_candidate(item, entity_id, profile):
    incoming = item["features"]
    agreements = {}
    conflicts = []
    support = []
    for field, value in incoming.items():
        values = profile.get(field, {})
        if value and value in values:
            agreements[field] = True
            support.append(values[value])
        elif value and values and field in {"website", "state", "postcode", "country"}:
            conflicts.append(f"different_{field}")

    name_similarity = 0.0
    name_support = None
    for field in ["legal_name", "trading_name"]:
        name = incoming.get(field)
        if not name:
            continue
        for other_field in ["legal_name", "trading_name"]:
            for known, provider in profile.get(other_field, {}).items():
                similarity = ratio(name, known) / 100
                if similarity > name_similarity:
                    name_similarity, name_support = similarity, provider
    if name_support:
        support.append(name_support)

    if incoming.get("legal_name") and profile.get("legal_name"):
        best_legal = max(ratio(incoming["legal_name"], name) / 100 for name in profile["legal_name"])
        if best_legal < 0.75:
            conflicts.append("different_legal_name")
    # A known different address requires review, even if another field agrees.
    if incoming.get("address") and profile.get("address") and not agreements.get("address"):
        conflicts.append("different_address")

    exact_name = name_similarity == 1.0
    full_address = incoming.get("address") or ""
    usable_address = len(full_address.split()) >= 3 and bool(re.search(r"[0-9]", full_address))
    score, method = 0.0, "insufficient_evidence"
    if exact_name and agreements.get("website"):
        score, method = 0.995, "name_website_exact"
    elif exact_name and agreements.get("address") and usable_address:
        score, method = (0.99 if agreements.get("legal_name") else 0.98), "name_address_exact"
    elif agreements.get("legal_name") and agreements.get("postcode") and agreements.get("locality"):
        score, method = 0.97, "multi_field_scored"
    elif name_similarity >= 0.97 and agreements.get("website"):
        score, method = 0.98, "multi_field_scored"
    elif name_similarity >= 0.99 and agreements.get("address") and usable_address and agreements.get("postcode"):
        score, method = 0.975, "multi_field_scored"

    if support:
        # Iteration can add evidence, but cannot increase its weakest link confidence.
        score = min(score, min(provider["link_confidence"] for provider in support))
    raw_score = score
    if conflicts:
        score = 0.0
    evidence = {
        "agreements": sorted(agreements), "name_similarity": round(name_similarity, 4),
        "conflicts": conflicts, "supporting_observations": support,
    }
    return {"entity_id": entity_id, "score": score, "raw_score": raw_score, "method": method, "evidence": evidence}


def choose_candidate(item, profiles, blocks):
    ids = candidate_entities(item, blocks)
    if len(ids) > MAX_CANDIDATES:
        return None, "too_many_candidates", []
    ranked = sorted(
        [score_candidate(item, entity_id, profiles[entity_id]) for entity_id in ids],
        key=lambda candidate: (-candidate["score"], candidate["entity_id"]),
    )
    # Postcode is a blocking key, not evidence that two businesses are related.
    ranked = [candidate for candidate in ranked
              if candidate["evidence"]["name_similarity"] >= 0.85
              or "website" in candidate["evidence"]["agreements"]
              or "address" in candidate["evidence"]["agreements"]]
    if not ranked:
        return None, "no_candidate", []
    best = ranked[0]
    if best["score"] < MIN_LINK_SCORE:
        reason = "conflicting_candidate" if any(c["evidence"]["conflicts"] for c in ranked) else "insufficient_evidence"
        return None, reason, ranked[:5]
    # An entity with contradictory evidence may still be the real alternative.
    # Don't silently discard it and select a superficially cleaner second entity.
    if any(c["raw_score"] >= MIN_LINK_SCORE and c["evidence"]["conflicts"] for c in ranked[1:]):
        return None, "ambiguous_conflicting_candidates", ranked[:5]
    competing = [c for c in ranked[1:] if c["score"] >= MIN_LINK_SCORE]
    if competing and best["score"] - competing[0]["score"] < MIN_SCORE_MARGIN:
        return None, "ambiguous_candidates", ranked[:5]
    return best, None, ranked[:5]


CANDIDATE_COLUMNS = [
    "source_id", "source_record_id", "gold_table", "record_name", "record_address", "record_website",
    "candidate_entity_id", "candidate_names", "candidate_addresses", "candidate_websites",
    "proposal_score", "automatic_link_score", "name_similarity", "agreements", "conflicts", "review_status",
]


def possible_match_report(observations, links):
    profiles, blocks = build_matching_index(observations, links)
    known_names = sorted({value for field, value in blocks if field == "name"})
    proposals = []
    for item in observations:
        if item["observation_id"] in links or item["problems"] or item["abn"] or item["acn"]:
            continue
        names = [item["features"][field] for field in ["legal_name", "trading_name"] if item["features"][field]]
        candidates = candidate_entities(item, blocks)
        # Broaden discovery for review only; the automatic rules above are unchanged.
        for name in names:
            for known, similarity, _ in process.extract(name, known_names, scorer=ratio, score_cutoff=85, limit=3):
                candidates.update(blocks[("name", known)])
        ranked = [score_candidate(item, entity_id, profiles[entity_id]) for entity_id in candidates]
        ranked = [candidate for candidate in ranked if candidate["evidence"]["name_similarity"] >= 0.85
                  or "website" in candidate["evidence"]["agreements"]
                  or "address" in candidate["evidence"]["agreements"]]
        ranked.sort(key=lambda candidate: (-candidate["raw_score"], -candidate["evidence"]["name_similarity"], candidate["entity_id"]))
        for candidate in ranked[:3]:
            profile = profiles[candidate["entity_id"]]
            evidence = candidate["evidence"]
            # A name suggestion receives a lower review score, never an automatic link.
            weak_score = 0.70 + 0.10 * evidence["name_similarity"]
            proposal_score = max(candidate["raw_score"], weak_score)
            proposals.append({
                "source_id": item["source_id"], "source_record_id": item["source_record_id"],
                "gold_table": item["gold_table"], "record_name": " / ".join(names),
                "record_address": item["features"]["address"], "record_website": item["features"]["website"],
                "candidate_entity_id": candidate["entity_id"],
                "candidate_names": " / ".join(sorted(set(profile.get("legal_name", {})) | set(profile.get("trading_name", {})))[:3]),
                "candidate_addresses": " / ".join(sorted(profile.get("address", {}))[:3]),
                "candidate_websites": " / ".join(sorted(profile.get("website", {}))[:3]),
                "proposal_score": round(proposal_score, 3), "automatic_link_score": candidate["score"],
                "name_similarity": evidence["name_similarity"],
                "agreements": ", ".join(evidence["agreements"]), "conflicts": ", ".join(evidence["conflicts"]),
                "review_status": "conflict — do not link" if evidence["conflicts"] else "possible match — needs review",
            })
    report = pd.DataFrame(proposals, columns=CANDIDATE_COLUMNS)
    return report.sort_values(["proposal_score", "source_id", "source_record_id"], ascending=[False, True, True]).reset_index(drop=True)


## Iterative entity resolution

In [7]:
REVIEW_COLUMNS = [
    "source_id", "source_record_id", "gold_table", "reason", "stated_abn", "stated_acn",
    "legal_name", "trading_name", "address", "website", "candidates",
]

def resolve_observations(observations):
    pool, links, issues = seed_identifiers(observations)
    history = [{"pass": 1, "new_links": len(links), "total_links": len(links), "stage": "identifiers"}]
    pending = [item for item in observations if item["observation_id"] not in links and item["observation_id"] not in issues]
    stopped_at_limit = False
    for pass_number in range(2, MAX_MATCH_PASSES + 2):
        profiles, blocks = build_matching_index(observations, links)
        additions = {}
        for item in pending:
            best, reason, candidates = choose_candidate(item, profiles, blocks)
            if best is not None:
                additions[item["observation_id"]] = make_link(
                    item, best["entity_id"], best["method"], best["score"], best["evidence"], pass_number,
                )
        links.update(additions)
        pending = [item for item in pending if item["observation_id"] not in additions]
        history.append({"pass": pass_number, "new_links": len(additions), "total_links": len(links), "stage": "matching"})
        if not additions or not pending:
            break
    else:
        stopped_at_limit = True

    # Show candidates against the final evidence, including the last iteration.
    profiles, blocks = build_matching_index(observations, links)
    review = []
    for item in observations:
        key = item["observation_id"]
        if key in links:
            continue
        if key in issues:
            reason, candidates = issues[key], []
        else:
            best, reason, candidates = choose_candidate(item, profiles, blocks)
            if best is not None and stopped_at_limit:
                reason = "iteration_limit"
        review.append({
            "source_id": item["source_id"], "source_record_id": item["source_record_id"],
            "gold_table": item["gold_table"], "reason": reason,
            "stated_abn": item["abn"], "stated_acn": item["acn"],
            "legal_name": item["features"]["legal_name"], "trading_name": item["features"]["trading_name"],
            "address": item["features"]["address"], "website": item["features"]["website"],
            "candidates": json.dumps(candidates, sort_keys=True),
        })
    # Identity inherited through a link is recorded as inferred, never written to gold.
    for link in links.values():
        entity = pool[link["entity_id"]]
        evidence = json.loads(link["match_evidence"])
        evidence["resolved_abn"] = entity["abn"]
        evidence["resolved_acn"] = entity["acn"]
        if evidence["stated_acn"] is None and entity["acn"] is not None:
            evidence["acn_origin"] = "inferred_from_entity"
        link["match_evidence"] = json.dumps(evidence, sort_keys=True)

    return (
        pd.DataFrame(list(pool.values()), columns=["entity_id", "abn", "acn"]),
        pd.DataFrame(list(links.values()), columns=LINK_COLUMNS),
        pd.DataFrame(review, columns=REVIEW_COLUMNS),
        pd.DataFrame(history),
    )

if DATABASE_READY:
    observations, duplicate_rows = prepare_observations(gold_observations)

    entities_pool, entity_link_index, unresolved_review, pass_metrics = resolve_observations(observations)

    assert entities_pool["entity_id"].is_unique

    assert entities_pool["abn"].dropna().is_unique

    assert entities_pool["acn"].dropna().is_unique

    assert not entity_link_index.duplicated(["source_id", "source_record_id"]).any()

    assert set(entity_link_index["entity_id"]) <= set(entities_pool["entity_id"])

    assert len(entity_link_index) + len(unresolved_review) == len(observations)

    stage("Resolve identity from ABN and ACN")

    for method, count in entity_link_index.groupby("match_method").size().items():
        if method in {"abn_exact", "acn_exact", "acn_derived_abn"}:
            print(f"  {method.replace('_', ' ')}: {count:,} linked records")

    print(f"Identity registry: {len(entities_pool):,} entities")

    print(f"Duplicate source rows collapsed: {duplicate_rows:,}")

    stage("Match by name, website and address")

    for _, row in pass_metrics[pass_metrics["stage"] == "matching"].iterrows():
        print(f"  Matching pass {int(row['pass']) - 1}: {int(row['new_links']):,} new links")

    inferred_count = (~entity_link_index["match_method"].isin(["abn_exact", "acn_exact", "acn_derived_abn"])).sum()

    print(f"Accepted field-based links: {inferred_count:,}; threshold: {MIN_LINK_SCORE:.2f}")

    if inferred_count == 0:
        print("No record passed the field-matching rules in this sample.")

    stage("Possible matches for human review")

    possible_matches = possible_match_report(observations, entity_link_index.set_index("link_id").to_dict("index"))

    possible_matches.to_csv(PART3_DIR / "possible_matches.csv", index=False)

    print(f"{len(possible_matches):,} candidate proposals for {possible_matches['source_record_id'].groupby(possible_matches['source_id']).nunique().sum():,} source records")

    print("These proposals are not accepted links. Name-only review scores are below 0.97.")

    if possible_matches.empty:
        print("No plausible name, website or address candidate was found in the available anchors.")
    else:
        print("Preview — source name beside candidate name:")
        display(possible_matches[["record_name", "candidate_names", "candidate_entity_id", "proposal_score", "conflicts"]].head(PREVIEW_ROWS))

    print("Full report:", PART3_DIR / "possible_matches.csv")

    stage("Resolution outcome by source")

    linked_keys = set(entity_link_index["link_id"])

    source_coverage = pd.DataFrame([
        {"gold_table": item["gold_table"], "linked": item["observation_id"] in linked_keys}
        for item in observations
    ]).groupby("gold_table")["linked"].agg(observations="count", linked="sum").reset_index()

    source_coverage["unresolved"] = source_coverage["observations"] - source_coverage["linked"]

    source_coverage.to_csv(PART3_DIR / "source_coverage.csv", index=False)

    for row in source_coverage.itertuples(index=False):
        print(f"  {dataset_label(row.gold_table)}: {row.linked:,} linked; {row.unresolved:,} unresolved")

    print(f"Total: {len(entity_link_index):,} linked; {len(unresolved_review):,} unresolved")

    if SHOW_DETAILS:
        display(pass_metrics)
        display(source_coverage)


PART 3 — Resolve identity from ABN and ACN
  abn exact: 3,909 linked records
  acn derived abn: 27 linked records
  acn exact: 23 linked records
Identity registry: 3,415 entities
Duplicate source rows collapsed: 0

PART 3 — Match by name, website and address
  Matching pass 1: 0 new links
Accepted field-based links: 0; threshold: 0.97
No record passed the field-matching rules in this sample.

PART 3 — Possible matches for human review
45 candidate proposals for 20 source records
These proposals are not accepted links. Name-only review scores are below 0.97.
Preview — source name beside candidate name:


,record_name,candidate_names,candidate_entity_id,proposal_score,conflicts
0,IMF AUSTRALIA LIMITED,IBM AUSTRALIA LIMITED,79000024733,0.795,
1,DDC PROPRIETARY LIMITED,CSR PROPRIETARY LIMITED,90000001276,0.791,
2,MAERSK AUSTRALIA PROPRIETARY LIMITED,OKUMA AUSTRALIA PROPRIETARY LIMITED,18000008211,0.790,
3,MAERSK AUSTRALIA PROPRIETARY LIMITED,NCR AUSTRALIA PROPRIETARY LIMITED,61000003592,0.790,
4,LGH HOLDINGS LIMITED,NGG HOLDINGS LIMITED,87000021518,0.790,


Full report: /Users/ario/Documents/firmable/outputs/part3/possible_matches.csv

PART 3 — Resolution outcome by source
  abn bulk extract: 1,000 linked; 0 unresolved
  acnc registered charities: 980 linked; 20 unresolved
  asic company: 986 linked; 14 unresolved
  business establishments location and industry classification: 0 linked; 1,000 unresolved
  victorian government schools abns: 993 linked; 7 unresolved
  victorian liquor licences by location: 0 linked; 1,000 unresolved
Total: 3,959 linked; 2,041 unresolved


## Persist tables and inspect results

In [8]:
def persist_part3(connection, entities, links, review, candidates=None):
    # Gold tables are never modified. These Part 3 tables are rebuilt on rerun.
    connection.register("_part3_entities", entities)
    connection.register("_part3_links", links)
    connection.register("_part3_review", review)
    if candidates is not None:
        connection.register("_part3_candidates", candidates)
    connection.execute("BEGIN TRANSACTION")
    try:
        connection.execute("DROP VIEW IF EXISTS part3_unlinked_review")
        connection.execute("DROP TABLE IF EXISTS entity_link_index")
        connection.execute("DROP TABLE IF EXISTS entities_pool")
        connection.execute("DROP TABLE IF EXISTS entity_resolution_review")
        connection.execute("""
            CREATE TABLE entities_pool (
                entity_id VARCHAR PRIMARY KEY,
                abn VARCHAR UNIQUE,
                acn VARCHAR UNIQUE
            )
        """)
        connection.execute("INSERT INTO entities_pool SELECT * FROM _part3_entities")
        connection.execute("""
            CREATE TABLE entity_link_index (
                link_id VARCHAR PRIMARY KEY,
                source_id VARCHAR NOT NULL,
                source_record_id VARCHAR NOT NULL,
                entity_id VARCHAR NOT NULL,
                match_method VARCHAR NOT NULL,
                match_evidence JSON NOT NULL,
                link_confidence DOUBLE NOT NULL,
                created_at TIMESTAMPTZ NOT NULL,
                UNIQUE(source_id, source_record_id)
            )
        """)
        connection.execute("INSERT INTO entity_link_index SELECT * FROM _part3_links")
        connection.execute("CREATE TABLE entity_resolution_review AS SELECT * FROM _part3_review")
        connection.execute("CREATE VIEW part3_unlinked_review AS SELECT * FROM entity_resolution_review")
        if candidates is not None:
            connection.execute("CREATE OR REPLACE TABLE entity_match_candidates AS SELECT * FROM _part3_candidates")
        connection.execute("COMMIT")
    except Exception:
        connection.execute("ROLLBACK")
        raise
    finally:
        for name in ["_part3_entities", "_part3_links", "_part3_review"]:
            connection.unregister(name)
        if candidates is not None:
            connection.unregister("_part3_candidates")

    for table in ["entities_pool", "entity_link_index", "entity_resolution_review"]:
        path = str(PART3_DIR / f"{table}.parquet").replace("'", "''")
        connection.execute(f"COPY {quote_identifier(table)} TO '{path}' (FORMAT PARQUET)")

if DATABASE_READY:
    stage("Save Part 3 results")

    database_written = False

    part3_con = connect_database(DB_PATH)

    if part3_con is not None:
        try:
            persist_part3(part3_con, entities_pool, entity_link_index, unresolved_review, possible_matches)
            database_written = True
        except duckdb.Error as error:
            print("Database save stopped safely; saving results as files:", error)
        finally:
            close_database(part3_con)
            part3_con = None

    for table, frame in [("entities_pool", entities_pool), ("entity_link_index", entity_link_index), ("entity_resolution_review", unresolved_review)]:
        frame.to_parquet(PART3_DIR / f"{table}.parquet", index=False)

    pass_metrics.to_csv(PART3_DIR / "resolution_passes.csv", index=False)

    run_metrics = {
        "input_mode": DATABASE_MODE, "database_written": database_written,
        "gold_rows": len(gold_observations), "unique_observations": len(observations),
        "duplicate_rows": duplicate_rows, "entities": len(entities_pool),
        "linked": len(entity_link_index), "unresolved": len(unresolved_review),
        "candidate_proposals": len(possible_matches),
        "candidate_records": len(possible_matches[["source_id", "source_record_id"]].drop_duplicates()),
        "matching_threshold": MIN_LINK_SCORE, "matching_margin": MIN_SCORE_MARGIN,
        "created_at": datetime.now(timezone.utc).isoformat(),
    }

    (PART3_DIR / "resolution_metrics.json").write_text(json.dumps(run_metrics, indent=2), encoding="utf-8")

    print("Destination:", DB_PATH if database_written else PART3_DIR)

    print("entities_pool: the identity registry")

    print("entity_link_index: accepted source-to-entity links")

    print("entity_resolution_review: unresolved observations and reasons")

    print("entity_match_candidates: possible matches requiring review")

    stage("Why records remain unresolved")

    for reason, count in unresolved_review.groupby("reason").size().items():
        print(f"  {reason.replace('_', ' ')}: {count:,} records")

    if SHOW_DETAILS:
        display(entities_pool.head(PREVIEW_ROWS))
        display(entity_link_index.head(PREVIEW_ROWS))
        display(unresolved_review.head(PREVIEW_ROWS))


PART 3 — Save Part 3 results
Destination: /Users/ario/Documents/firmable/firmable.duckdb
entities_pool: the identity registry
entity_link_index: accepted source-to-entity links
entity_resolution_review: unresolved observations and reasons
entity_match_candidates: possible matches requiring review

PART 3 — Why records remain unresolved
  abn acn conflict: 14 records
  no candidate: 2,027 records


## Random review sample of 50 links

In [9]:
def make_review_sample(links, observations, path):
    rows = {item["observation_id"]: item for item in observations}
    sampled = links.sort_values("link_id").sample(n=min(REVIEW_SAMPLE_SIZE, len(links)), random_state=REVIEW_SEED).copy()
    sampled["gold_table"] = sampled["link_id"].map(lambda key: rows[key]["gold_table"])
    for field in ["legal_name", "trading_name", "address", "website"]:
        sampled[field] = sampled["link_id"].map(lambda key: rows[key]["features"].get(field))
    sampled["stated_abn"] = sampled["link_id"].map(lambda key: rows[key]["abn"])
    sampled["stated_acn"] = sampled["link_id"].map(lambda key: rows[key]["acn"])
    sampled["claim_fingerprint"] = sampled["link_id"].map(
        lambda key: hashlib.sha256(json.dumps(rows[key]["claim"], sort_keys=True).encode()).hexdigest()
    )
    sampled["label_correct"] = ""
    sampled["reviewer_notes"] = ""
    if path.exists():
        old = pd.read_csv(path, dtype=str, keep_default_na=False).drop_duplicates("link_id", keep="last").set_index("link_id")
        for index, row in sampled.iterrows():
            if row["link_id"] not in old.index:
                continue
            previous = old.loc[row["link_id"]]
            unchanged = all(str(previous.get(column, "")) == str(row[column])
                            for column in ["entity_id", "claim_fingerprint", "match_method", "match_evidence"])
            if unchanged:
                for column in ["label_correct", "reviewer_notes"]:
                    sampled.at[index, column] = previous.get(column, "")
    sampled.to_csv(path, index=False)
    return sampled

if DATABASE_READY:
    review_sample = make_review_sample(entity_link_index, observations, REVIEW_PATH)

    stage("Review the accuracy of accepted links")

    print(f"Prepared {len(review_sample)} accepted links for checking.")

    print("Question: does each source record belong to the entity we assigned?")

    print("This is an accuracy audit. Labels do not change the entity links.")

    print("Review file:", REVIEW_PATH)

    print("Use true, false or unsure in label_correct; existing unchanged labels are retained.")

    if SHOW_DETAILS:
        display(review_sample[["entity_id", "match_method", "legal_name", "stated_abn", "stated_acn", "label_correct"]].head(PREVIEW_ROWS))


PART 3 — Review the accuracy of accepted links
Prepared 50 accepted links for checking.
Question: does each source record belong to the entity we assigned?
This is an accuracy audit. Labels do not change the entity links.
Review file: /Users/ario/Documents/firmable/outputs/part3/link_review_50.csv
Use true, false or unsure in label_correct; existing unchanged labels are retained.


In [10]:


if DATABASE_READY:
    stage("Accepted link methods")

    for method, count in entity_link_index.groupby("match_method").size().items():
        print(f"  {method.replace('_', ' ')}: {count:,}")


PART 3 — Accepted link methods
  abn exact: 3,909
  acn derived abn: 27
  acn exact: 23


## What happened, and what remains

The identifier step creates the initial entities. The iterative field matcher can then inherit an existing `entity_id` when names, websites and addresses agree strongly enough. Possible-match proposals are shown separately from accepted links.

In the initial 6,000-row run, identifiers produced 3,959 links to 3,415 entities. Field matching produced zero accepted links. The 2,041 unresolved records consisted of 14 ABN/ACN conflicts and 2,027 records without a usable anchored match. Each source contributes only its Part 2 test extraction; matching cannot find an entity whose identifying observations are absent from that sample.

The possible-match report now also searches similar names across anchors. Those lower-confidence suggestions remain review-only, with contradictions shown explicitly. The automatic threshold is unchanged.

Before expanding automatic matching: inspect proposals; review true/false links; fix identifier conflicts in the source mappings; and expand relevant ABR/ASIC gold coverage so missing businesses can become anchors. Lowering a threshold alone does not supply missing evidence.

## Optional human review with context

Set `REVIEW_LINKS_NOW = True` to check the random sample during the notebook run. Each question shows the source, identifiers, target entity, method and other linked evidence. These decisions measure accepted-link accuracy; they do not approve new candidate links.

In [11]:
def review_links(path, entities, links, observations):
    review = pd.read_csv(path, dtype=str, keep_default_na=False)
    registry = entities.set_index("entity_id")
    records = {item["observation_id"]: item for item in observations}
    for index, sampled in review.iterrows():
        if sampled["label_correct"] in {"true", "false"}:
            continue
        stage(f"Check accepted link {index + 1} of {len(review)}")
        item = records[sampled["link_id"]]
        target = registry.loc[sampled["entity_id"]]
        print("Source:", dataset_label(sampled["gold_table"]))
        print("Source record:", sampled["source_record_id"])
        print("Source name:", item["features"]["legal_name"] or item["features"]["trading_name"] or "not supplied")
        print("Source ABN:", item["abn"] or "not supplied", "| ACN:", item["acn"] or "not supplied")
        print("Source address:", item["features"]["address"] or "not supplied")
        print("Source website:", item["features"]["website"] or "not supplied")
        print("Assigned entity:", sampled["entity_id"])
        print("Registry ABN:", text_value(target["abn"]) or "not stated", "| ACN:", text_value(target["acn"]) or "not stated")
        print("Link method:", sampled["match_method"].replace("_", " "))
        print("Rule score:", sampled["link_confidence"], "— not measured accuracy")
        evidence = json.loads(sampled["match_evidence"])
        print("Fields agreeing:", ", ".join(evidence.get("agreements", [])) or "identifier-based link")
        print("Conflicts:", ", ".join(evidence.get("conflicts", [])) or "none detected")
        other = links[(links["entity_id"] == sampled["entity_id"]) & (links["link_id"] != sampled["link_id"])]
        print("Other observations linked to this entity:", len(other))
        for key in other["link_id"].head(3):
            known = records[key]
            print("  ", dataset_label(known["gold_table"]), "—", known["features"]["legal_name"] or known["features"]["trading_name"] or "name not supplied")
        print("Question: does this source record belong to the assigned entity, based on the evidence above?")
        print("1 - Correct: mark this link supported")
        print("2 - Incorrect: flag this link as wrong for follow-up")
        print("3 - Unsure: leave correctness unconfirmed")
        print("Enter - Stop: retain saved review answers")
        while True:
            choice = input(f"Check entity {sampled['entity_id']} [1/2/3, Enter stop]: ").strip()
            if not choice:
                return review
            if choice in {"1", "2", "3"}:
                review.at[index, "label_correct"] = {"1": "true", "2": "false", "3": "unsure"}[choice]
                review.at[index, "reviewer_notes"] = input("Reason or correction (optional): ").strip()
                review.to_csv(path, index=False)
                break
            print("Choose Correct, Incorrect or Unsure using 1, 2 or 3.")
    return review

REVIEW_LINKS_NOW = False

if DATABASE_READY:
    if REVIEW_LINKS_NOW:
        review_sample = review_links(REVIEW_PATH, entities_pool, entity_link_index, observations)

    stage("Measured link accuracy")

    labels = pd.read_csv(REVIEW_PATH, dtype=str, keep_default_na=False)

    labelled = labels[labels["label_correct"].str.lower().isin(["true", "false"])]

    if labelled.empty:
        print("Precision is pending: no links have true/false review labels yet.")
    else:
        correct = labelled["label_correct"].str.lower().eq("true").sum()
        print(f"Review progress: {len(labelled)} of {len(labels)} sampled links have true/false labels")
        print(f"{correct} correct out of {len(labelled)} reviewed; measured precision: {correct / len(labelled):.3f}")
        print("This measures the reviewed accepted links, not the unlinked candidate proposals.")


PART 3 — Measured link accuracy
Precision is pending: no links have true/false review labels yet.


## Part 4 — Entity profiles and field conflicts

Implemented in `04_entity_profiles.ipynb` using the accepted links from this notebook:

1. Collect linked gold observations for each existing `entity_id` and preserve provenance.
2. For each field claim, calculate **C = R × L × F**, keeping all three components.
3. When normalized values agree, sum their scores and cap final field confidence at 1.
4. When values differ, leave the profile field and its confidence null. Preserve all claims and report each alternative's support divided by total support.
5. Save `firmable_entities`, uncontested field scores, all field claims, field conflicts and alternative scores. AI decision, Data analyser decision and final verdict stay null for future review.

Run Part 4 after this notebook. Unlinked records and possible matches remain here until accepted entity links exist. Part 4 does not create links or automatically choose conflict winners.